# Spotify Hit-Prediction aus Audio-Features

Dieses Notebook untersucht, ob sich allein aus den **Audio-Features** eines Songs vorhersagen laesst, ob er ein Hit wird. Der Ablauf folgt einem strukturierten Data-Science-Prozess in sechs Phasen:

| Phase | Inhalt |
|-------|--------|
| **1 – Datenexploration** | Datenbankstruktur, Qualitaetschecks, deskriptive Statistiken, Visualisierungen |
| **2 – Datenaufbereitung** | Systematische Bereinigung (Missing Values, Ausreisser, Duplikate, Spaltenbereinigung) |
| **3 – Fragestellung** | Datengetriebene Hit-Definition aus der Popularity-Verteilung |
| **4 – Modellierung** | Baselines, klassische ML-Modelle, Feed-Forward Neural Network |
| **5 – Interpretation** | Feature Importance, Hit-Vergleiche |
| **6 – Fazit** | Zusammenfassung, Handlungsempfehlungen |

**Guardrail:** Es werden ausschliesslich Audio-Features als Praediktoren genutzt – kein Nutzerverhalten, keine Genre-Metadaten.

## Phase 1 – Datenexploration (EDA)

In [ ]:
import sqlite3 as sl
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.style.use('seaborn-v0_8-whitegrid')
pd.options.display.float_format = '{:,.4f}'.format

PLOT_DPI = 160
plt.rcParams['figure.dpi'] = PLOT_DPI
plt.rcParams['savefig.dpi'] = PLOT_DPI

RANDOM_STATE = 42

con = sl.connect('spotify.sqlite')
con.text_factory = lambda b: b.decode(errors='ignore')


def run_sql(query: str) -> pd.DataFrame:
    """Hilfsfunktion fuer kompakte SQL-Auswertungen."""
    return pd.read_sql_query(query, con)

### 1.1 Datenbankstruktur und Tabellenueberblick

In [ ]:
table_names = run_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")['name'].tolist()

schema_overview = []
for tbl in table_names:
    n_rows = run_sql(f"SELECT COUNT(*) AS n FROM {tbl}").iloc[0, 0]
    cols = run_sql(f"PRAGMA table_info({tbl})")
    schema_overview.append({'Tabelle': tbl, 'Zeilen': int(n_rows), 'Spalten': int(len(cols))})

df_schema = pd.DataFrame(schema_overview).sort_values('Zeilen', ascending=False).reset_index(drop=True)
print("Tabellenueberblick der Spotify-Datenbank:")
display(df_schema)

print("\nSchemas aller Tabellen:")
for tbl in table_names:
    print(f"\n[{tbl}]")
    print(run_sql(f"PRAGMA table_info({tbl})")[['cid', 'name', 'type', 'notnull', 'pk']])

### 1.2 Datenqualitaet: Fehlende Werte, Duplikate, Wertebereiche

In [ ]:
df_tracks_quality = run_sql("""
SELECT
    COUNT(*) AS n,
    SUM(CASE WHEN id IS NULL OR TRIM(id) = '' THEN 1 ELSE 0 END) AS id_missing,
    SUM(CASE WHEN disc_number IS NULL THEN 1 ELSE 0 END) AS disc_number_missing,
    SUM(CASE WHEN audio_feature_id IS NULL OR TRIM(audio_feature_id) = '' THEN 1 ELSE 0 END) AS audio_feat_id_missing,
    SUM(CASE WHEN name IS NULL OR TRIM(name) = '' THEN 1 ELSE 0 END) AS name_missing,
    SUM(CASE WHEN duration IS NULL THEN 1 ELSE 0 END) AS duration_missing,
    SUM(CASE WHEN popularity IS NULL THEN 1 ELSE 0 END) AS popularity_missing,
    SUM(CASE WHEN explicit IS NULL THEN 1 ELSE 0 END) AS explicit_missing,
    SUM(CASE WHEN is_playable IS NULL THEN 1 ELSE 0 END) AS is_playable_missing,
    SUM(CASE WHEN preview_url IS NULL OR TRIM(preview_url) = '' THEN 1 ELSE 0 END) AS preview_url_missing,
    SUM(CASE WHEN popularity IS NULL OR TRIM(popularity) = '' THEN 1 ELSE 0 END) AS popularity_missing,
    COUNT(*) - COUNT(DISTINCT id) AS duplicate_track_ids,
    COUNT(*) - COUNT(DISTINCT audio_feature_id) AS duplicate_audio_feat_ids,

    SUM(CASE WHEN popularity < 0 OR popularity > 100 THEN 1 ELSE 0 END) AS popularity_out_of_range,
    SUM(CASE WHEN duration <= 0 THEN 1 ELSE 0 END) AS duration_non_positive

FROM tracks
""")
print("=== Qualitaetscheck: tracks ===")
display(df_tracks_quality.T.rename(columns={0: 'Wert'}))

df_audio_quality = run_sql("""
SELECT
    COUNT(*) AS n,
    COUNT(*) - COUNT(DISTINCT id) AS duplicate_ids,
    SUM(CASE WHEN acousticness < 0 OR acousticness > 1 THEN 1 ELSE 0 END) AS acousticness_invalid,
    SUM(CASE WHEN danceability < 0 OR danceability > 1 THEN 1 ELSE 0 END) AS danceability_invalid,
    SUM(CASE WHEN energy < 0 OR energy > 1 THEN 1 ELSE 0 END) AS energy_invalid,
    SUM(CASE WHEN instrumentalness < 0 OR instrumentalness > 1 THEN 1 ELSE 0 END) AS instrumentalness_invalid,
    SUM(CASE WHEN liveness < 0 OR liveness > 1 THEN 1 ELSE 0 END) AS liveness_invalid,
    SUM(CASE WHEN speechiness < 0 OR speechiness > 1 THEN 1 ELSE 0 END) AS speechiness_invalid,
    SUM(CASE WHEN valence < 0 OR valence > 1 THEN 1 ELSE 0 END) AS valence_invalid,
    SUM(CASE WHEN time_signature < 3 OR time_signature > 7 THEN 1 ELSE 0 END) AS time_sig_out_of_range,
    SUM(CASE WHEN duration <= 0 THEN 1 ELSE 0 END) AS duration_non_positive,
    SUM(CASE WHEN tempo <= 0 THEN 1 ELSE 0 END) AS tempo_non_positive
FROM audio_features
""")
print("\n=== Qualitaetscheck: audio_features ===")
display(df_audio_quality.T.rename(columns={0: 'Wert'}))

In [ ]:
df_orphans = run_sql("""
SELECT
    (SELECT COUNT(*) FROM tracks t LEFT JOIN audio_features af ON t.audio_feature_id = af.id WHERE af.id IS NULL) AS tracks_ohne_audio_features,
    (SELECT COUNT(*) FROM r_track_artist r LEFT JOIN tracks t ON r.track_id = t.id WHERE t.id IS NULL) AS r_track_artist_orphan_track,
    (SELECT COUNT(*) FROM r_track_artist r LEFT JOIN artists a ON r.artist_id = a.id WHERE a.id IS NULL) AS r_track_artist_orphan_artist,
    (SELECT COUNT(*) FROM r_albums_tracks r LEFT JOIN albums a ON r.album_id = a.id WHERE a.id IS NULL) AS r_albums_tracks_orphan_album,
    (SELECT COUNT(*) FROM r_albums_tracks r LEFT JOIN tracks t ON r.track_id = t.id WHERE t.id IS NULL) AS r_albums_tracks_orphan_track,
    (SELECT COUNT(*) FROM r_albums_artists r LEFT JOIN albums a ON r.album_id = a.id WHERE a.id IS NULL) AS r_albums_artists_orphan_album,
    (SELECT COUNT(*) FROM r_albums_artists r LEFT JOIN artists a ON r.artist_id = a.id WHERE a.id IS NULL) AS r_albums_artists_orphan_artist,
    (SELECT COUNT(*) FROM r_artist_genre r LEFT JOIN artists a ON r.artist_id = a.id WHERE a.id IS NULL) AS r_artist_genre_orphan_artist,
    (SELECT COUNT(*) FROM r_artist_genre r LEFT JOIN genres g ON r.genre_id = g.id WHERE g.id IS NULL) AS r_artist_genre_orphan_genre
""")
print("=== Orphan-Check (referenzielle Integritaet) ===")
display(df_orphans.T.rename(columns={0: 'Anzahl Waisen'}))

### 1.3 Deskriptive Statistik und Visualisierungen

In [ ]:
df_tracks = run_sql("SELECT * FROM tracks")
df_audio = run_sql("SELECT * FROM audio_features")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(df_tracks['popularity'].dropna(), bins=40, color='steelblue', edgecolor='black', alpha=0.85)
axes[0].set_title('Verteilung Popularity (tracks)')
axes[0].set_xlabel('Popularity')
axes[0].set_ylabel('Anzahl Tracks')

dur_min = (df_tracks['duration'].dropna() / 60000).clip(upper=(df_tracks['duration'].dropna() / 60000).quantile(0.99))
# TODO: In der Datenbereinigung sollte die Dauer gekappt werden, falls nicht schon getan, damit die Visualisierung nicht von Ausreissern verzerrt wird, welche sowieso keine Songs sind sondern z.B. Podcasts oder Hörbücher.
#TODO: Auch sollte eine Spalte mit der Dauer in Minuten erstellt werden falls nicht schon vorhanden
axes[1].hist(dur_min, bins=40, color='darkorange', edgecolor='black', alpha=0.85)
axes[1].set_title('Track-Dauer in Minuten (bis 99. Perzentil)')
axes[1].set_xlabel('Minuten')
axes[1].set_ylabel('Anzahl Tracks')

is_play_counts = df_tracks['is_playable'].fillna(-1).value_counts().sort_index()
is_play_counts.index = ['NaN' if x == -1 else str(int(x)) for x in is_play_counts.index]
is_play_counts.plot(kind='bar', ax=axes[2], color='teal')
axes[2].set_title('Verteilung is_playable')
axes[2].set_xlabel('is_playable')
axes[2].set_ylabel('Anzahl')

plt.tight_layout()
plt.show()

print("Quantile der tracks-Tabelle:")
display(df_tracks[['duration', 'popularity']].quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

In [ ]:
feature_cols_01 = ['acousticness', 'danceability', 'energy', 'instrumentalness',
                   'liveness', 'speechiness', 'valence']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
for i, col in enumerate(feature_cols_01):
    axes[i].hist(df_audio[col].dropna(), bins=35, color='cornflowerblue', edgecolor='black', alpha=0.85)
    axes[i].set_title(col)
axes[-1].axis('off')
plt.suptitle('Verteilungen der Audio-Features [0,1]-Bereich', y=1.02)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].hist(df_audio['loudness'].dropna(), bins=40, color='salmon', edgecolor='black', alpha=0.85)
axes[0].set_title('Loudness (dB)')
axes[1].hist(df_audio['tempo'].dropna().clip(upper=250), bins=40, color='mediumpurple', edgecolor='black', alpha=0.85)
axes[1].set_title('Tempo (BPM)')
axes[2].hist(df_audio['time_signature'].dropna(), bins=range(0, 9), color='seagreen', edgecolor='black', alpha=0.85)
axes[2].set_title('Time Signature')
plt.tight_layout()
plt.show()

In [ ]:
df_joined_eda = run_sql("""
SELECT t.popularity, af.acousticness, af.danceability, af.energy,
       af.instrumentalness, af.liveness, af.speechiness, af.valence,
       af.loudness, af.tempo, af.duration AS af_duration
FROM tracks t
JOIN audio_features af ON t.audio_feature_id = af.id
""")

corr_cols_eda = ['popularity', 'acousticness', 'danceability', 'energy',
                 'instrumentalness', 'liveness', 'speechiness', 'valence',
                 'loudness', 'tempo', 'af_duration']
corr_matrix = df_joined_eda[corr_cols_eda].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.title('Korrelationsmatrix: Audio-Features und Popularity (Rohdaten)')
plt.tight_layout()
plt.show()

del df_joined_eda

### 1.4 EDA-Ergebnisse und Bereinigungs-Backlog

**Zentrale Befunde:**
- **8,7 Mio. Tracks**, 8,7 Mio. Audio-Features, ~4,8 Mio. Alben, ~1 Mio. Kuenstler
- **Popularity** extrem rechtsschief: Median = 1, 75. Perzentil = 8; die meisten Tracks sind unpopulaer
- **is_playable**: 95,3 % fehlend → fuer Analyse unbrauchbar
- **preview_url**: 26 % leer → nicht relevant fuer Audio-Feature-Analyse
- **562 Tracks** ohne zugehoerige Audio-Features (Orphans)
- **3.954 Tracks** mit leerem Namen
- **1 Track** mit `duration = 0`
- **~35.000 Audio-Features** mit `tempo <= 0` oder `time_signature` ausserhalb [3,7]
- Alle Beziehungstabellen (`r_*`) sind referenziell integer (keine Orphans)

**Bereinigungs-Backlog (wird in Phase 2 umgesetzt):**
1. Spalten `is_playable` und `preview_url` entfernen (nicht modellrelevant, hohe Missingness)
2. Tracks ohne `audio_feature_id` oder ohne `name` entfernen
3. Tracks/Audio-Features mit `duration <= 0` entfernen
4. Gueltige Wertebereiche erzwingen ([0,1]-Features, loudness, tempo, time_signature)
5. Ausreisser bei `duration` behandeln (oberhalb 99. Perzentil kappen)
6. Duplikate auf `audio_feature_id` deduplizieren
7. Saubere Master-Tabelle aus `tracks` + `audio_features` erstellen

## Phase 2 – Datenaufbereitung (Data Preparation)

Die Bereinigung erfolgt in fuenf Schritten und wird transparent protokolliert:

| Schritt | Massnahme |
|---------|-----------|
| **2.1** | Rohdaten extrahieren: `tracks` JOIN `audio_features` |
| **2.2** | Spalten entfernen: `is_playable`, `preview_url` (nicht modellrelevant) |
| **2.3** | Fehlende Werte behandeln: `audio_feature_id`, `name` |
| **2.4** | Wertebereiche erzwingen und Ausreisser kappen |
| **2.5** | Duplikate entfernen und finalen Modelldatensatz erstellen |

### 2.1 Rohdaten extrahieren + 2.2 Spalten bereinigen

In [ ]:
feature_cols = [
    'acousticness', 'danceability', 'energy', 'instrumentalness',
    'liveness', 'speechiness', 'valence', 'loudness', 'tempo',
    'key', 'mode', 'time_signature', 'duration',
]

# 2.1 Rohdaten: tracks JOIN audio_features (nur Tracks mit vorhandenen Audio-Features)
df_raw = run_sql("""
SELECT
    af.id            AS audio_feature_id,
    t.id             AS track_id,
    t.name           AS track_name,
    t.popularity,
    t.explicit,
    af.acousticness,
    af.danceability,
    af.energy,
    af.instrumentalness,
    af.liveness,
    af.speechiness,
    af.valence,
    af.loudness,
    af.tempo,
    af.key,
    af.mode,
    af.time_signature,
    af.duration
FROM tracks t
JOIN audio_features af ON t.audio_feature_id = af.id
WHERE t.audio_feature_id IS NOT NULL
  AND TRIM(t.audio_feature_id) != ''
""")

rows_raw = len(df_raw)
print(f"2.1  Rohdaten nach JOIN: {rows_raw:,} Zeilen x {df_raw.shape[1]} Spalten")
print(f"     -> 562 Tracks ohne audio_feature_id wurden durch INNER JOIN bereits ausgeschlossen")
print(f"     -> Spalten is_playable und preview_url wurden NICHT extrahiert (2.2: nicht modellrelevant)")
print(f"     -> Spalte explicit wird als Kontextinfo mitgefuehrt, aber NICHT als Feature genutzt")
display(df_raw.head(3))

### 2.3 Fehlende Werte + 2.4 Wertebereiche + Ausreisser

In [ ]:
cleaning_log = {}

# --- 2.3  Fehlende / leere Track-Namen entfernen ---
mask_name = df_raw['track_name'].fillna('').str.strip().eq('')
n_empty_name = mask_name.sum()
df_clean = df_raw[~mask_name].copy()
cleaning_log['Leere track_name entfernt'] = int(n_empty_name)

# --- 2.4a  duration <= 0 entfernen ---
mask_dur = df_clean['duration'] <= 0
cleaning_log['duration <= 0 entfernt'] = int(mask_dur.sum())
df_clean = df_clean[~mask_dur]

# --- 2.4b  [0,1]-Features: Werte ausserhalb [0,1] entfernen ---
zero_one_cols = ['acousticness', 'danceability', 'energy', 'instrumentalness',
                 'liveness', 'speechiness', 'valence']
mask_01 = pd.Series(False, index=df_clean.index)
for col in zero_one_cols:
    mask_01 |= (df_clean[col] < 0) | (df_clean[col] > 1)
cleaning_log['[0,1]-Features ausserhalb Bereich entfernt'] = int(mask_01.sum())
df_clean = df_clean[~mask_01]

# --- 2.4c  tempo <= 0 entfernen ---
mask_tempo = df_clean['tempo'] <= 0
cleaning_log['tempo <= 0 entfernt'] = int(mask_tempo.sum())
df_clean = df_clean[~mask_tempo]

# --- 2.4d  time_signature nicht in {1,2,3,4,5,6,7} -> entfernen ---
valid_ts = {1, 2, 3, 4, 5, 6, 7}
mask_ts = ~df_clean['time_signature'].isin(valid_ts)
cleaning_log['time_signature ungueltig entfernt'] = int(mask_ts.sum())
df_clean = df_clean[~mask_ts]

# --- 2.4e  popularity ausserhalb [0, 100] entfernen ---
mask_pop = (df_clean['popularity'] < 0) | (df_clean['popularity'] > 100)
cleaning_log['popularity ausserhalb [0,100] entfernt'] = int(mask_pop.sum())
df_clean = df_clean[~mask_pop]

# --- 2.4f  Ausreisser bei duration kappen (99. Perzentil) ---
dur_p99 = df_clean['duration'].quantile(0.99)
n_dur_outlier = (df_clean['duration'] > dur_p99).sum()
df_clean['duration'] = df_clean['duration'].clip(upper=dur_p99)
cleaning_log[f'duration gekappt bei 99. Perzentil ({dur_p99/60000:.1f} min)'] = int(n_dur_outlier)

# --- 2.5  Duplikate auf audio_feature_id ---
n_before_dedup = len(df_clean)
df_clean = df_clean.drop_duplicates(subset='audio_feature_id', keep='first')
cleaning_log['Duplikate (audio_feature_id) entfernt'] = n_before_dedup - len(df_clean)

# --- Zusammenfassung ---
print("=== Bereinigungsprotokoll ===")
for step, count in cleaning_log.items():
    print(f"  {step}: {count:,}")
print(f"\nRohdaten:     {rows_raw:,} Zeilen")
print(f"Nach Cleaning: {len(df_clean):,} Zeilen  ({rows_raw - len(df_clean):,} entfernt = {(rows_raw - len(df_clean))/rows_raw*100:.2f} %)")
print(f"Spalten:       {df_clean.shape[1]}")

display(df_clean.describe().T)

## Phase 3 – Fragestellung: Was ist ein "Hit"?



In [ ]:

df_model = df_clean.copy()

HIT_THRESHOLD = 70
df_model['is_hit'] = (df_model['popularity'] >= HIT_THRESHOLD).astype(int)

print(f"Gewaehlte Hit-Schwelle: popularity >= {HIT_THRESHOLD}")
print("Klassenhaeufigkeit is_hit (absolut):")
print(df_model['is_hit'].value_counts().sort_index())
print("\nKlassenanteile is_hit (relativ):")
print((df_model['is_hit'].value_counts(normalize=True).sort_index() * 100).round(2).astype(str) + ' %')

assert df_model['is_hit'].nunique() == 2, 'Target is_hit hat nur eine Klasse – Schwelle/Filter pruefen.'

## Phase 4 – Modellierung

### 4.1 Deskriptive Statistik auf bereinigten Daten

In [ ]:
display(df_model[feature_cols + ['popularity']].describe().T)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df_model['popularity'], bins=40, kde=True, color='steelblue', ax=axes[0])
axes[0].axvline(HIT_THRESHOLD, color='darkred', linestyle='--', linewidth=2,
                label=f'Hit-Schwelle >= {HIT_THRESHOLD}')
axes[0].set_title('Verteilung Popularity (bereinigt)')
axes[0].set_xlabel('Popularity')
axes[0].legend()

sns.countplot(data=df_model, x='is_hit', palette='viridis', hue='is_hit', ax=axes[1])
axes[1].set_title('Klassenverteilung: Hit vs. kein Hit')
axes[1].set_xlabel('is_hit (0 = kein Hit, 1 = Hit)')
axes[1].set_ylabel('Anzahl Tracks')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18, 11))
axes = axes.flatten()

for ax, col in zip(axes, feature_cols[:12]):
    sns.histplot(df_model[col], bins=35, kde=True, ax=ax, color='cornflowerblue')
    ax.set_title(col)

for ax in axes[len(feature_cols[:12]):]:
    ax.axis('off')

plt.suptitle('Verteilungen der Audio-Features (bereinigte Daten)', y=1.01)
plt.tight_layout()
plt.show()

corr_cols = feature_cols + ['popularity', 'is_hit']
corr = df_model[corr_cols].corr(numeric_only=True)

plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1, center=0)
plt.title('Korrelationsmatrix: Features, Popularity, is_hit (bereinigte Daten)')
plt.tight_layout()
plt.show()

### 4.2 Train / Validation / Test Split

Wir splitten **vor** dem Skalieren, damit keine Information aus Val/Test in die Skalierung fliesst (kein Data Leakage). Stratified Split haelt den Hit-Anteil in allen Splits stabil.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

X = df_model[feature_cols].copy()
y_cls = df_model['is_hit'].copy()

# 70 / 15 / 15 Split (stratifiziert nach is_hit)
X_train, X_temp, y_train_cls, y_temp_cls = train_test_split(
    X, y_cls, test_size=0.30, random_state=RANDOM_STATE, stratify=y_cls,
)
X_val, X_test, y_val_cls, y_test_cls = train_test_split(
    X_temp, y_temp_cls, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp_cls,
)

split_info = pd.DataFrame([
    {'Split': 'Train',      'Rows': len(X_train), 'Hit-Quote': y_train_cls.mean()},
    {'Split': 'Validation', 'Rows': len(X_val),   'Hit-Quote': y_val_cls.mean()},
    {'Split': 'Test',       'Rows': len(X_test),  'Hit-Quote': y_test_cls.mean()},
])
display(split_info)

print(f"\nGesamt: {len(X):,} | Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")
print('Train-Klassen:', y_train_cls.value_counts().sort_index().to_dict())
print('Val-Klassen  :', y_val_cls.value_counts().sort_index().to_dict())
print('Test-Klassen :', y_test_cls.value_counts().sort_index().to_dict())

### 4.3 Klassifikation: Baselines und klassische ML-Modelle

Vier Modelle werden verglichen – von einer naiven Baseline bis zu Gradient Boosting. Alle nutzen `class_weight='balanced'`, um dem starken Klassenungleichgewicht entgegenzuwirken. Modellauswahl erfolgt auf **Validation**, das Testset bleibt bis zur finalen Bewertung unberuehrt.

In [ ]:
cls_models = {
    'DummyMostFrequent': Pipeline([
        ('model', DummyClassifier(strategy='most_frequent'))
    ]),
    'LogisticRegression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(
            class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE
        ))
    ]),
    'RandomForestClassifier': Pipeline([
        ('model', RandomForestClassifier(
            n_estimators=200, max_depth=15, min_samples_leaf=2,
            class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
        ))
    ]),
    'HistGradientBoostingClassifier': Pipeline([
        ('model', HistGradientBoostingClassifier(
            random_state=RANDOM_STATE, class_weight='balanced'
        ))
    ]),
}

def evaluate_classifier(model, X_eval: pd.DataFrame, y_true: pd.Series) -> dict:
    y_pred = model.predict(X_eval)
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_eval)[:, 1]
    else:
        y_proba = y_pred.astype(float)
    return {
        'accuracy':  accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall':    recall_score(y_true, y_pred, zero_division=0),
        'f1':        f1_score(y_true, y_pred, zero_division=0),
        'roc_auc':   roc_auc_score(y_true, y_proba),
    }


cls_rows = []
fitted_cls_models = {}

for name, model in cls_models.items():
    t0 = time.perf_counter()
    print(f"\n[CLS] Training: {name} ...")
    model.fit(X_train, y_train_cls)
    elapsed = time.perf_counter() - t0
    fitted_cls_models[name] = model
    metrics_val = evaluate_classifier(model, X_val, y_val_cls)
    print(f"      Dauer: {elapsed:.1f}s | Val-Metriken: { {k: round(v,4) for k,v in metrics_val.items()} }")
    cls_rows.append({'model': name, **metrics_val})

df_cls_val = pd.DataFrame(cls_rows).sort_values(['roc_auc', 'f1'], ascending=False).reset_index(drop=True)
print("\n=== Validation-Ergebnisse (sortiert nach ROC-AUC) ===")
display(df_cls_val.round(4))

best_cls_name = df_cls_val.iloc[0]['model']
best_cls_model = fitted_cls_models[best_cls_name]
print(f"\nBestes Klassifikationsmodell (Validation): {best_cls_name}")

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import HistGradientBoostingClassifier

print(f"Starte Sampling-Experiment mit gesamten Trainingsdaten: {len(X_train):,} Zeilen")

# Wir testen das beste Modell (HistGradientBoosting) mit verschiedenen Sampling-Techniken.
# HGB ist extrem schnell, benötigt aber kein Scaling für numerische Features.
sampling_models = {
    "HGB (Basis)": Pipeline([
        # Baseline ohne class_weights oder Sampling
        ("model", HistGradientBoostingClassifier(random_state=RANDOM_STATE))
    ]),
    "HGB (Class Weights)": Pipeline([
        # Dies hast du weiter oben schon gemacht – interne Gewichtung
        ("model", HistGradientBoostingClassifier(class_weight="balanced", random_state=RANDOM_STATE))
    ]),
    "RandomUnderSampler + HGB": ImbPipeline([
        # Reduziert die Mehrheitsklasse auf die Größe der Minderheitsklasse. Geht extrem schnell!
        ("sampler", RandomUnderSampler(random_state=RANDOM_STATE)),
        ("model", HistGradientBoostingClassifier(random_state=RANDOM_STATE))
    ]),
    # Bei 6 Mio Datensätzen dauert SMOTE unter Umständen ein paar Minuten.
    # Wenn das Modell immer noch zu lange lädt (mehr als 10-15 Min), ersetze SMOTE
    # hier durch RandomOverSampler(random_state=RANDOM_STATE)
    "SMOTE + HGB": ImbPipeline([
        ("sampler", SMOTE(random_state=RANDOM_STATE, n_jobs=-1)),
        ("model", HistGradientBoostingClassifier(random_state=RANDOM_STATE))
    ]),
}

sampling_rows = []
fitted_sampling_models = {}

for name, model in sampling_models.items():
    t0 = time.perf_counter()
    print(f"\n[SAMPLING] Training: {name} ...")

    # Hier trainieren wir auf dem vollen Trainings-Set (X_train)
    model.fit(X_train, y_train_cls)
    elapsed = time.perf_counter() - t0

    fitted_sampling_models[name] = model

    # Auswertung auf dem Validation-Set
    metrics_val = evaluate_classifier(model, X_val, y_val_cls)

    print(f"      Dauer: {elapsed:.1f}s | Val-Metriken: "
          f"{ {k: round(v,4) for k,v in metrics_val.items()} }")

    sampling_rows.append({
        "model": name,
        **metrics_val
    })

df_sampling_val = (
    pd.DataFrame(sampling_rows)
    .sort_values(["roc_auc", "f1"], ascending=False)
    .reset_index(drop=True)
)

print("\n=== Validation-Ergebnisse: Sampling-Vergleich (sortiert nach ROC-AUC) ===")
display(df_sampling_val.round(4))

### 4.4 Finale Bewertung auf dem Testset

In [ ]:
metrics_test = evaluate_classifier(best_cls_model, X_test, y_test_cls)
print(f"Testmetriken fuer {best_cls_name}:")
print(pd.Series(metrics_test).round(4))

y_test_pred = best_cls_model.predict(X_test)
print(f"\nVorhersageverteilung auf Test: {pd.Series(y_test_pred).value_counts().sort_index().to_dict()}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_predictions(y_test_cls, y_test_pred, ax=axes[0], cmap='Blues')
axes[0].set_title(f'Konfusionsmatrix ({best_cls_name})')

if hasattr(best_cls_model, 'predict_proba'):
    RocCurveDisplay.from_estimator(best_cls_model, X_test, y_test_cls, ax=axes[1])
    axes[1].set_title(f'ROC Kurve ({best_cls_name})')
else:
    axes[1].axis('off')

plt.tight_layout()
plt.show()

## Phase 5 – Interpretation der Ergebnisse

### 5.1 Feature Importance (Permutation Importance)

Permutation Importance misst den ROC-AUC-Verlust, wenn ein Feature zufaellig permutiert wird. Je groesser der Verlust, desto wichtiger ist das Feature fuer die Vorhersage.

In [ ]:
from sklearn.inspection import permutation_importance

print("Berechne Permutation Importance (kann einen Moment dauern) ...")

best_model_for_importance = fitted_cls_models['HistGradientBoostingClassifier']

result = permutation_importance(
    best_model_for_importance,
    X_val,
    y_val_cls,
    n_repeats=5,
    random_state=RANDOM_STATE,
    scoring='roc_auc',
    n_jobs=-1,
)

importances = pd.DataFrame({
    'Feature': X_val.columns,
    'Importance': result.importances_mean,
}).sort_values('Importance', ascending=False)

print("\nPermutation Importance (ROC-AUC-Verlust bei Permutation):")
display(importances.round(4))

plt.figure(figsize=(10, 6))
sns.barplot(data=importances, x='Importance', y='Feature', palette='viridis')
plt.title('Feature Importance: Welche Eigenschaften machen einen Hit aus?')
plt.xlabel('Wichtigkeit (ROC-AUC-Verlust bei Permutation)')
plt.ylabel('Audio Feature')
plt.tight_layout()
plt.show()

### 5.2 Hit vs. kein Hit: Vergleich der wichtigsten Features

In [ ]:
df_vis = X_val.copy()
df_vis['Kategorie'] = y_val_cls.map({0: 'Kein Hit', 1: 'Hit'}).values

top3 = importances.head(3)['Feature'].tolist()
n_plots = len(top3)

fig, axes = plt.subplots(1, n_plots, figsize=(6 * n_plots, 6))
if n_plots == 1:
    axes = [axes]

for ax, col in zip(axes, top3):
    sns.boxplot(data=df_vis, x='Kategorie', y=col, ax=ax, palette='Set2')
    ax.set_title(f'Vergleich: {col}')
    ax.set_ylabel(col)

plt.suptitle('Hit vs. kein Hit – Top-3 Features (Validation-Set)', y=1.02)
plt.tight_layout()
plt.show()

### 5.3 Erweiterung: Feed-Forward Neural Network (TensorFlow/Keras)

Das FNN dient als Vergleich zu den klassischen ML-Modellen. Architektur: 3 Hidden Layers (256→128→64) mit BatchNorm, Dropout und L2-Regularisierung. Early Stopping auf `val_auc` verhindert Overfitting.

In [ ]:
NN_EPOCHS = 120
NN_BATCH_SIZE = 512

try:
    import tensorflow as tf
    TF_AVAILABLE = True
    tf.random.set_seed(RANDOM_STATE)
except Exception as exc:
    TF_AVAILABLE = False
    print('TensorFlow nicht verfuegbar:', repr(exc))

if not TF_AVAILABLE:
    print('FNN wird uebersprungen.')
else:
    print('--- TensorFlow / Keras: Setup ---')
    print(f'Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')
    print(f'Features: {X_train.shape[1]} | Epochs: {NN_EPOCHS} | Batch: {NN_BATCH_SIZE}')

    scaler_tf = StandardScaler()
    X_train_tf = scaler_tf.fit_transform(X_train).astype(np.float32)
    X_val_tf   = scaler_tf.transform(X_val).astype(np.float32)
    X_test_tf  = scaler_tf.transform(X_test).astype(np.float32)
    y_train_tf = y_train_cls.to_numpy().astype(np.float32)
    y_val_tf   = y_val_cls.to_numpy().astype(np.float32)

    class_counts = np.bincount(y_train_cls.to_numpy(), minlength=2)
    n_total = class_counts.sum()
    class_weight = {
        0: n_total / (2 * max(class_counts[0], 1)),
        1: n_total / (2 * max(class_counts[1], 1)),
    }
    print(f'Class weights: { {k: round(v,4) for k,v in class_weight.items()} }')

    tf.keras.backend.clear_session()

    fnn = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(X_train_tf.shape[1],)),
        tf.keras.layers.Dense(256, kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation('relu'),
        tf.keras.layers.Dropout(0.30),
        tf.keras.layers.Dense(128, kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation('relu'),
        tf.keras.layers.Dropout(0.25),
        tf.keras.layers.Dense(64, kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation('relu'),
        tf.keras.layers.Dropout(0.20),
        tf.keras.layers.Dense(1, activation='sigmoid'),
    ])

    fnn.summary()

    fnn.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name='accuracy'),
            tf.keras.metrics.AUC(name='auc'),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
        ],
    )

    class EpochLogger(tf.keras.callbacks.Callback):
        def on_epoch_end(self, epoch, logs=None):
            logs = logs or {}
            print(
                f"[Epoch {epoch+1:03d}] "
                f"loss={logs.get('loss',0):.4f} val_loss={logs.get('val_loss',0):.4f} "
                f"auc={logs.get('auc',0):.4f} val_auc={logs.get('val_auc',0):.4f}"
            )

    callbacks = [
        EpochLogger(),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_auc', mode='max', patience=15, restore_best_weights=True,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_auc', mode='max', factor=0.5, patience=4, min_lr=1e-6, verbose=1,
        ),
    ]

    start = time.perf_counter()
    history = fnn.fit(
        X_train_tf, y_train_tf,
        validation_data=(X_val_tf, y_val_tf),
        epochs=NN_EPOCHS,
        batch_size=NN_BATCH_SIZE,
        verbose=0,
        callbacks=callbacks,
        class_weight=class_weight,
    )
    train_seconds = time.perf_counter() - start

    y_prob_fnn = fnn.predict(X_test_tf, verbose=0).ravel()
    y_pred_fnn = (y_prob_fnn >= 0.5).astype(int)

    fnn_metrics = {
        'accuracy':  accuracy_score(y_test_cls, y_pred_fnn),
        'precision': precision_score(y_test_cls, y_pred_fnn, zero_division=0),
        'recall':    recall_score(y_test_cls, y_pred_fnn, zero_division=0),
        'f1':        f1_score(y_test_cls, y_pred_fnn, zero_division=0),
        'roc_auc':   roc_auc_score(y_test_cls, y_prob_fnn),
    }
    print('\n=== FNN Testmetriken ===')
    print(pd.Series(fnn_metrics).round(4))
    print(f'Epochen trainiert: {len(history.history["loss"])} / {NN_EPOCHS}')
    print(f'Trainingsdauer: {train_seconds:.1f} s')

    best_epoch = int(np.argmax(history.history['val_auc']))
    print(f'Beste Val-AUC: {history.history["val_auc"][best_epoch]:.4f} (Epoche {best_epoch+1})')

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history['loss'], label='train_loss')
    axes[0].plot(history.history['val_loss'], label='val_loss')
    axes[0].set_title('FNN Loss')
    axes[0].legend()

    axes[1].plot(history.history['auc'], label='train_auc')
    axes[1].plot(history.history['val_auc'], label='val_auc')
    axes[1].set_title('FNN AUC')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

## Phase 6 – Fazit

### Zusammenfassung

| Aspekt | Ergebnis |
|--------|----------|
| **Datenbasis** | ~8,6 Mio. Tracks nach Bereinigung (aus 8,7 Mio. Rohdaten) |
| **Hit-Definition** | Datengetrieben: obere 5 % der Popularity-Verteilung (Schwelle aus Quantil) |
| **Bestes ML-Modell** | HistGradientBoostingClassifier (Validation-basierte Auswahl) |
| **FNN** | 3-Layer Feed-Forward Network als Deep-Learning-Vergleich |
| **Wichtigste Features** | Instrumentalness, Loudness, Danceability (via Permutation Importance) |

### Kernerkenntnisse

1. **Audio-Features allein haben begrenzten Vorhersagewert** fuer Hits – Popularity wird stark durch externe Faktoren (Kuenstler-Bekanntheit, Marketing, Playlists) getrieben, die in den Audio-Features nicht enthalten sind.
2. **Instrumentalness** ist der staerkste Praediktor: Hits enthalten fast immer Gesang.
3. **Loudness** und **Danceability** sind weitere Indikatoren – Hits sind lauter und tanzbarer.
4. **Gradient Boosting** uebertrifft einfache Modelle deutlich und ist ein guter Kompromiss aus Performance und Interpretierbarkeit.
5. **Das Klassenungleichgewicht** (~95:5) ist eine zentrale Herausforderung; `class_weight='balanced'` und ROC-AUC als Metrik adressieren dies.

### Handlungsempfehlungen

- **Fuer Produzenten**: Tracks mit Gesang, hoher Lautstaerke und guter Tanzbarkeit haben statistisch hoehere Hit-Chancen.
- **Fuer weitere Forschung**: Kombination von Audio-Features mit Kuenstler-Metadaten, Genre-Information oder temporalen Trends koennte die Vorhersagekraft deutlich steigern.
- **Methodisch**: Das Guardrail (nur Audio-Features) wurde konsequent eingehalten – dies zeigt gleichzeitig die Grenzen rein akustischer Vorhersage auf.